# 🏠 House Price Predictor — Model Training & Evaluation

This notebook covers:
1. Data preparation
2. Baseline model (Linear Regression)
3. Random Forest
4. XGBoost with hyperparameter tuning
5. Model comparison
6. Feature importance
7. Predicted vs Actual analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

from data_processing import prepare_dataset

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

## 1. Prepare Data

In [ ]:
X, y, encoders = prepare_dataset('../data/train.csv')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')
print(f'Features:         {X.shape[1]}')
X.head()

## 2. Baseline — Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print(f'Linear Regression RMSE: ${rmse(y_test, lr_preds):,.0f}')
print(f'Linear Regression R²:   {r2_score(y_test, lr_preds):.4f}')

## 3. Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print(f'Random Forest RMSE: ${rmse(y_test, rf_preds):,.0f}')
print(f'Random Forest R²:   {r2_score(y_test, rf_preds):.4f}')

## 4. XGBoost with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_base = XGBRegressor(random_state=42, verbosity=0)
gs = GridSearchCV(xgb_base, param_grid, cv=3,
                  scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)

print(f'\nBest params: {gs.best_params_}')
best_xgb = gs.best_estimator_
xgb_preds = best_xgb.predict(X_test)

print(f'\nXGBoost RMSE: ${rmse(y_test, xgb_preds):,.0f}')
print(f'XGBoost R²:   {r2_score(y_test, xgb_preds):.4f}')

## 5. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost (tuned)'],
    'RMSE': [
        rmse(y_test, lr_preds),
        rmse(y_test, rf_preds),
        rmse(y_test, xgb_preds)
    ],
    'R²': [
        r2_score(y_test, lr_preds),
        r2_score(y_test, rf_preds),
        r2_score(y_test, xgb_preds)
    ]
}).sort_values('RMSE')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2563eb', '#1a1612', '#c0392b']

results.plot(kind='bar', x='Model', y='RMSE', ax=axes[0], color=colors, legend=False)
axes[0].set_title('RMSE by Model (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE ($)')
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

results.plot(kind='bar', x='Model', y='R²', ax=axes[1], color=colors, legend=False)
axes[1].set_title('R² Score by Model (higher is better)', fontweight='bold')
axes[1].set_ylabel('R² Score')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

results.style.format({'RMSE': '${:,.0f}', 'R²': '{:.4f}'})

## 6. Cross-Validation (XGBoost)

In [ ]:
cv_scores = cross_val_score(best_xgb, X, y, cv=5, scoring='neg_root_mean_squared_error')
rmse_scores = -cv_scores

print('5-Fold Cross-Validation Results (XGBoost):')
for i, s in enumerate(rmse_scores):
    print(f'  Fold {i+1}: ${s:,.0f}')
print(f'\nMean RMSE: ${rmse_scores.mean():,.0f} ± ${rmse_scores.std():,.0f}')

## 7. Feature Importance

In [ ]:
importances = best_xgb.feature_importances_
feat_df = pd.Series(importances, index=X.columns).sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
feat_df.plot(kind='barh', color='#2563eb', alpha=0.85)
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features (XGBoost)', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter
axes[0].scatter(y_test, xgb_preds, alpha=0.4, color='#2563eb', s=20)
max_val = max(y_test.max(), xgb_preds.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=1.5, label='Perfect')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test - xgb_preds
axes[1].scatter(xgb_preds, residuals, alpha=0.4, color='#c0392b', s=20)
axes[1].axhline(0, color='black', lw=1, ls='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot', fontweight='bold')

plt.suptitle('XGBoost Prediction Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mean residual: ${residuals.mean():,.0f}')
print(f'Std residual:  ${residuals.std():,.0f}')

## Summary

| Model              | RMSE         | R²     |
|--------------------|-------------|--------|
| Linear Regression  | see above   | ~0.81  |
| Random Forest      | see above   | ~0.88  |
| XGBoost (tuned)    | **best**    | **~0.90** |

XGBoost performs best and is saved to `models/xgboost_model.pkl` via `src/train.py`.

➡️ Run `uvicorn main:app --reload` to serve the model via REST API.